# LeetCode 925 — Long Pressed Name bằng Python

**Link đề:** https://leetcode.com/problems/long-pressed-name/
**Độ khó:** Easy · **Chủ đề:** Two Pointers (hai con trỏ), String

```python
class Solution:
    def isLongPressedName(self, name: str, typed: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu đúng mô hình "giữ phím": mỗi ký tự của `name` được gõ **1 hoặc nhiều lần liên tiếp**.
2. Học kỹ thuật **nén đoạn lặp** (run-length encoding) và `itertools.groupby`.
3. Viết lời giải **hai con trỏ** chạy song song trên hai chuỗi — cách chuẩn phỏng vấn.
4. Một cách sáng tạo: dựng **biểu thức chính quy** (regex) từ `name`.
5. Trace, chứng minh tính đúng, Time/Space, các lỗi kinh điển (ký tự thừa cuối chuỗi, thiếu ký tự lặp).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — có phải tên bạn mình gõ bị giữ phím?

Bạn của bạn gõ tên `name`. Khi gõ một ký tự `c`, phím có thể bị **giữ lâu**, khiến `c` xuất hiện **1 hoặc nhiều lần**. Kết quả trên màn hình là `typed`.

Trả `True` nếu `typed` **có thể** là tên `name` với vài ký tự (có thể không ký tự nào) bị giữ phím.

| `name` | `typed` | Output | Giải thích |
|---|---|:---:|---|
| `"alex"` | `"aaleex"` | `True` | `a` và `e` bị giữ |
| `"saeed"` | `"ssaaedd"` | `False` | `"ee"` cần ít nhất 2 chữ `e`, typed chỉ có 1 |
| `"alex"` | `"alex"` | `True` | không giữ phím nào |
| `"alex"` | `"alexxr"` | `False` | `r` thừa, không có trong tên |

**Constraints:** `1 <= name.length, typed.length <= 1000`; chỉ chữ thường.

Câu hỏi then chốt: "giữ phím" chỉ **thêm** ký tự giống ký tự **vừa gõ** — không bao giờ xóa ký tự, không thêm ký tự mới.

In [1]:
# Mô phỏng: gõ "alex" với số lần nhấn mỗi ký tự
name = "alex"
presses = [2, 1, 2, 1]          # 'a' 2 lần, 'l' 1 lần, 'e' 2 lần, 'x' 1 lần
typed = "".join(ch * k for ch, k in zip(name, presses))
print(typed)

aaleex


`ch * k` lặp chuỗi `ch` đúng `k` lần (`"a" * 3 == "aaa"`). `zip(name, presses)` ghép ký tự với số lần nhấn.

## 2. Nền tảng Python cần dùng

### 2.1. Duyệt chuỗi bằng chỉ số và so sánh ký tự kề nhau

`s[i]` là ký tự thứ `i`; `s[i] == s[i-1]` kiểm tra ký tự hiện tại có giống ký tự trước. Cẩn thận `i = 0`: `s[-1]` trong Python là **ký tự cuối** (chỉ số âm đếm từ cuối), không phải lỗi — dễ gây bug ngầm!

In [2]:
s = "aaleex"
for i in range(len(s)):
    same = i > 0 and s[i] == s[i - 1]
    print(i, s[i], "giống ký tự trước" if same else "")
print("s[-1] =", s[-1], "<- chỉ số âm lấy từ cuối, không báo lỗi!")

0 a 
1 a giống ký tự trước
2 l 
3 e 
4 e giống ký tự trước
5 x 
s[-1] = x <- chỉ số âm lấy từ cuối, không báo lỗi!


### 2.2. Nén đoạn lặp (run-length encoding — RLE)

Biến chuỗi thành danh sách cặp `(ký tự, độ dài đoạn)`:
`"aaleex"` → `[('a',2), ('l',1), ('e',2), ('x',1)]`.

Viết tay bằng vòng `while` với hai chỉ số `i` (đầu đoạn) và `j` (chạy tới hết đoạn).

In [3]:
def rle(s: str) -> list[tuple[str, int]]:
    groups = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        groups.append((s[i], j - i))
        i = j
    return groups

print(rle("aaleex"))
print(rle("saeed"))
print(rle("ssaaedd"))

[('a', 2), ('l', 1), ('e', 2), ('x', 1)]
[('s', 1), ('a', 1), ('e', 2), ('d', 1)]
[('s', 2), ('a', 2), ('e', 1), ('d', 2)]


### 2.3. `itertools.groupby` — nén đoạn có sẵn

`groupby(s)` gom các phần tử **liên tiếp giống nhau**, trả về cặp `(key, iterator_của_nhóm)`. `len(list(g))` cho độ dài đoạn.

Lưu ý: `groupby` chỉ gom phần tử **kề nhau**, không phải gom toàn bộ như `Counter`.

In [4]:
from itertools import groupby

print([(k, len(list(g))) for k, g in groupby("aaleex")])
print([(k, len(list(g))) for k, g in groupby("abba")], "<- 'a' xuất hiện 2 nhóm tách biệt")

[('a', 2), ('l', 1), ('e', 2), ('x', 1)]
[('a', 1), ('b', 2), ('a', 1)] <- 'a' xuất hiện 2 nhóm tách biệt


### 2.4. Regex cơ bản với `re.fullmatch`

Biểu thức chính quy (regular expression) mô tả "mẫu" chuỗi:
- `a+` nghĩa là "một hoặc nhiều chữ `a`".
- `re.fullmatch(pattern, s)` trả về đối tượng khớp nếu **toàn bộ** `s` khớp mẫu, ngược lại `None`.
- `re.escape(c)` để ký tự đặc biệt được hiểu đúng nghĩa đen (ở bài này chỉ có chữ thường, nhưng là thói quen tốt).

In [5]:
import re
print(re.fullmatch("a+l+", "aaal") is not None)    # True
print(re.fullmatch("a+l+", "aaalx") is not None)   # False: thừa x
print(re.fullmatch("a+l+", "l") is not None)       # False: thiếu a

True
False
False


## 3. Cách trực giác — so sánh hai bản nén

Ý tưởng: nén cả `name` và `typed` thành các đoạn.
- `typed` hợp lệ ⇔ **cùng số đoạn**, đoạn thứ `k` có **cùng ký tự**, và độ dài đoạn trong `typed` **≥** độ dài đoạn trong `name`.

Ví dụ `saeed` → `s1 a1 e2 d1`; `ssaaedd` → `s2 a2 e1 d2`. Đoạn `e`: typed có 1 < 2 → False.

In [6]:
def long_pressed_rle(name: str, typed: str) -> bool:
    gn, gt = rle(name), rle(typed)
    if len(gn) != len(gt):
        return False
    for (cn, kn), (ct, kt) in zip(gn, gt):
        if cn != ct or kt < kn:
            return False
    return True

for name, typed in [("alex", "aaleex"), ("saeed", "ssaaedd"), ("alex", "alex"), ("alex", "alexxr")]:
    print(f"{name!r:8} {typed!r:10} -> {long_pressed_rle(name, typed)}")

'alex'   'aaleex'   -> True
'saeed'  'ssaaedd'  -> False
'alex'   'alex'     -> True
'alex'   'alexxr'   -> False


`for (cn, kn), (ct, kt) in zip(gn, gt)` — mở gói **lồng nhau**: mỗi phần tử của `zip` là cặp hai tuple, mỗi tuple lại mở thành (ký tự, độ dài).

### 3.1. Phiên bản `groupby`

In [7]:
def long_pressed_groupby(name: str, typed: str) -> bool:
    gn = [(k, len(list(g))) for k, g in groupby(name)]
    gt = [(k, len(list(g))) for k, g in groupby(typed)]
    return len(gn) == len(gt) and all(cn == ct and kt >= kn for (cn, kn), (ct, kt) in zip(gn, gt))

print(long_pressed_groupby("alex", "aaleex"), long_pressed_groupby("saeed", "ssaaedd"))

True False


## 4. Nút thắt và insight

Cách nén đúng và `O(n + m)`, nhưng tạo hai danh sách đoạn (bộ nhớ `O(n + m)`). Ta có thể **so sánh trực tiếp** khi duyệt, không cần lưu?

**Insight — mỗi ký tự `typed[j]` chỉ có thể là một trong hai thứ:**
1. **Ký tự kế tiếp của tên**: `typed[j] == name[i]` → cả hai tiến (`i += 1`, `j += 1`).
2. **Ký tự bị giữ phím**: `typed[j] == typed[j-1]` (lặp lại ký tự vừa gõ) → chỉ `j` tiến.
3. Không phải cả hai → **không hợp lệ** ngay.

Cuối cùng phải dùng **hết** `name`: `i == len(name)`.

**Thứ tự ưu tiên quan trọng:** thử khớp `name[i]` **trước**, vì với `name = "aab"` và `typed = "aab"`, chữ `a` thứ hai phải được tính là ký tự của tên, không phải giữ phím.

In [8]:
# Vì sao phải ưu tiên khớp name trước? Thử ưu tiên "giữ phím" trước:
def wrong_priority(name, typed):
    i = 0
    for j in range(len(typed)):
        if j > 0 and typed[j] == typed[j - 1]:
            continue                       # coi là giữ phím trước
        if i < len(name) and name[i] == typed[j]:
            i += 1
        else:
            return False
    return i == len(name)

print("wrong_priority('aab','aab') =", wrong_priority("aab", "aab"), "| đúng là True")

wrong_priority('aab','aab') = False | đúng là True


## 5. Lời giải chính — hai con trỏ

**Mảnh 1:** `i = 0` (con trỏ trên `name`).

**Mảnh 2:** duyệt `j` qua `typed`.

**Mảnh 3:** quyết định cho `typed[j]`:
- `if i < len(name) and name[i] == typed[j]: i += 1`
- `elif j > 0 and typed[j] == typed[j - 1]: pass` (giữ phím, bỏ qua)
- `else: return False`

**Mảnh 4:** `return i == len(name)`.

Chú ý điều kiện `i < len(name)` đứng **trước** `name[i]` — nhờ **đánh giá ngắn mạch** (short-circuit) của `and`, khi `i` đã hết thì Python không đọc `name[i]` nữa → không `IndexError`.

In [9]:
# Short-circuit: vế phải không được tính khi vế trái False
name = "ab"
i = 2
print(i < len(name) and name[i] == "x")   # False, không lỗi

False


In [10]:
def long_pressed_two_pointers(name: str, typed: str) -> bool:
    i = 0
    for j in range(len(typed)):
        if i < len(name) and name[i] == typed[j]:
            i += 1
        elif j > 0 and typed[j] == typed[j - 1]:
            continue
        else:
            return False
    return i == len(name)

for name, typed in [("alex", "aaleex"), ("saeed", "ssaaedd"), ("alex", "alex"), ("alex", "alexxr"), ("aab", "aab")]:
    print(f"{name!r:8} {typed!r:10} -> {long_pressed_two_pointers(name, typed)}")

'alex'   'aaleex'   -> True
'saeed'  'ssaaedd'  -> False
'alex'   'alex'     -> True
'alex'   'alexxr'   -> False
'aab'    'aab'      -> True


## 6. TRACE từng bước

Ví dụ `name = "saeed"`, `typed = "ssaaedd"`.

| j | typed[j] | i | name[i] | Khớp name? | Giữ phím? | Hành động |
|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 0 | s | 0 | s | ✓ | | i=1 |
| 1 | s | 1 | a | ✗ | s==s ✓ | bỏ qua |
| 2 | a | 1 | a | ✓ | | i=2 |
| 3 | a | 2 | e | ✗ | a==a ✓ | bỏ qua |
| 4 | e | 2 | e | ✓ | | i=3 |
| 5 | d | 3 | e | ✗ | d≠e ✗ | **return False** |

Tại `j = 5`, tên đang chờ chữ `e` thứ hai nhưng typed đã sang `d` → sai.

In [11]:
def trace_long_pressed(name: str, typed: str) -> bool:
    print(f"name={name!r}, typed={typed!r}")
    i = 0
    for j in range(len(typed)):
        want = name[i] if i < len(name) else "∅"
        if i < len(name) and name[i] == typed[j]:
            i += 1
            act = f"khớp name -> i={i}"
        elif j > 0 and typed[j] == typed[j - 1]:
            act = "giữ phím -> bỏ qua"
        else:
            print(f"  j={j} typed[j]={typed[j]!r} chờ {want!r}: không khớp, không giữ phím -> False")
            return False
        print(f"  j={j} typed[j]={typed[j]!r} chờ {want!r}: {act}")
    print(f"  hết typed: i={i}, len(name)={len(name)} -> {i == len(name)}")
    return i == len(name)

trace_long_pressed("saeed", "ssaaedd")
print()
trace_long_pressed("alex", "aaleex")

name='saeed', typed='ssaaedd'
  j=0 typed[j]='s' chờ 's': khớp name -> i=1
  j=1 typed[j]='s' chờ 'a': giữ phím -> bỏ qua
  j=2 typed[j]='a' chờ 'a': khớp name -> i=2
  j=3 typed[j]='a' chờ 'e': giữ phím -> bỏ qua
  j=4 typed[j]='e' chờ 'e': khớp name -> i=3
  j=5 typed[j]='d' chờ 'e': không khớp, không giữ phím -> False

name='alex', typed='aaleex'
  j=0 typed[j]='a' chờ 'a': khớp name -> i=1
  j=1 typed[j]='a' chờ 'l': giữ phím -> bỏ qua
  j=2 typed[j]='l' chờ 'l': khớp name -> i=2
  j=3 typed[j]='e' chờ 'e': khớp name -> i=3
  j=4 typed[j]='e' chờ 'x': giữ phím -> bỏ qua
  j=5 typed[j]='x' chờ 'x': khớp name -> i=4
  hết typed: i=4, len(name)=4 -> True


True

In [12]:
# Trường hợp typed hết trước khi dùng hết name
trace_long_pressed("alex", "aale")

name='alex', typed='aale'
  j=0 typed[j]='a' chờ 'a': khớp name -> i=1
  j=1 typed[j]='a' chờ 'l': giữ phím -> bỏ qua
  j=2 typed[j]='l' chờ 'l': khớp name -> i=2
  j=3 typed[j]='e' chờ 'e': khớp name -> i=3
  hết typed: i=3, len(name)=4 -> False


False

## 7. Các lời giải khác và bảng so sánh

### 7.1. Hai con trỏ "đi theo đoạn" (so độ dài đoạn tại chỗ)

Tại mỗi bước, đếm độ dài đoạn hiện tại trong cả `name` và `typed` (không lưu danh sách), so sánh ngay. Đây là bản "RLE không dùng bộ nhớ".

In [13]:
def long_pressed_run_compare(name: str, typed: str) -> bool:
    i = j = 0
    n, m = len(name), len(typed)
    while i < n and j < m:
        if name[i] != typed[j]:
            return False
        c = name[i]
        i0, j0 = i, j
        while i < n and name[i] == c:
            i += 1
        while j < m and typed[j] == c:
            j += 1
        if j - j0 < i - i0:          # đoạn trong typed ngắn hơn
            return False
    return i == n and j == m

for name, typed in [("alex", "aaleex"), ("saeed", "ssaaedd"), ("alex", "alexxr"), ("aab", "aab")]:
    print(f"{name!r:8} {typed!r:10} -> {long_pressed_run_compare(name, typed)}")

'alex'   'aaleex'   -> True
'saeed'  'ssaaedd'  -> False
'alex'   'alexxr'   -> False
'aab'    'aab'      -> True


`i = j = 0` là **gán chuỗi**: cả `i` và `j` cùng nhận `0`.

### 7.2. Regex dựng từ `name`

Mỗi đoạn `(c, k)` của `name` → mẫu `c{k,}` nghĩa là "chữ `c` ít nhất `k` lần". Ghép lại và `fullmatch` với `typed`.

Ví dụ `saeed` → `s{1,}a{1,}e{2,}d{1,}`.

In [14]:
def long_pressed_regex(name: str, typed: str) -> bool:
    pattern = "".join(f"{re.escape(c)}{{{k},}}" for c, k in rle(name))
    return re.fullmatch(pattern, typed) is not None

print("pattern cho 'saeed':", "".join(f"{c}{{{k},}}" for c, k in rle("saeed")))
for name, typed in [("alex", "aaleex"), ("saeed", "ssaaedd"), ("alex", "alexxr")]:
    print(f"{name!r:8} {typed!r:10} -> {long_pressed_regex(name, typed)}")

pattern cho 'saeed': s{1,}a{1,}e{2,}d{1,}
'alex'   'aaleex'   -> True
'saeed'  'ssaaedd'  -> False
'alex'   'alexxr'   -> False


Trong f-string, `{{` và `}}` in ra dấu ngoặc nhọn **thật**; `{k}` vẫn là chỗ thay giá trị. Vì vậy `f"{c}{{{k},}}"` với `c='e', k=2` cho ra `e{2,}`.

Mẹo nhỏ: mẫu `c+` lặp `k` lần (`e+e+`) cũng tương đương `e{2,}`.

### 7.3. Bảng so sánh

`n = len(name)`, `m = len(typed)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| RLE tự viết + so sánh | `O(n + m)` | `O(n + m)` | dễ hiểu, tách 2 bước |
| `groupby` | `O(n + m)` | `O(n + m)` | ngắn gọn |
| **Hai con trỏ (khớp / giữ phím / sai)** | `O(n + m)` | `O(1)` | chuẩn phỏng vấn |
| Hai con trỏ theo đoạn | `O(n + m)` | `O(1)` | rõ ý "so độ dài đoạn" |
| Regex `c{k,}` | thường `O(n + m)` với mẫu này | `O(n)` cho pattern | sáng tạo, phụ thuộc engine regex |

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def isLongPressedName(self, name: str, typed: str) -> bool:
        i = 0
        for j in range(len(typed)):
            if i < len(name) and name[i] == typed[j]:
                i += 1
            elif j > 0 and typed[j] == typed[j - 1]:
                continue
            else:
                return False
        return i == len(name)

sol = Solution()
assert sol.isLongPressedName("alex", "aaleex") == True
assert sol.isLongPressedName("saeed", "ssaaedd") == False
print("Solution PASS 2 ví dụ đề bài")

Solution PASS 2 ví dụ đề bài


## 9. Vì sao thuật toán đúng?

**Invariant:** sau khi xử lý `typed[0..j]` mà chưa trả `False`, thì `typed[0..j]` có thể được tạo ra bằng cách gõ `name[0..i-1]` với giữ phím, và ký tự cuối được gõ là `name[i-1]` (= `typed[j]`).

- **Khớp name:** `typed[j] == name[i]` → gõ thêm ký tự tên, `i` tăng — invariant giữ.
- **Giữ phím:** `typed[j] == typed[j-1] == name[i-1]` → ký tự lặp của phím vừa gõ — invariant giữ.
- **Còn lại:** `typed[j]` khác cả ký tự tên kế tiếp lẫn ký tự vừa gõ → không cách nào sinh ra nó → `False` đúng.

**Vì sao ưu tiên khớp name là an toàn (tham lam đúng)?** Nếu `typed[j]` vừa bằng `name[i]` vừa bằng `typed[j-1]`, nghĩa là `name[i] == name[i-1]` (hai ký tự liền nhau giống nhau trong tên). Dùng `typed[j]` cho `name[i]` không bao giờ tệ hơn: ký tự lặp sau đó vẫn có thể được coi là giữ phím của `name[i]` (cùng chữ). Ngược lại, coi nó là giữ phím có thể khiến thiếu chữ cho `name[i]` (phản ví dụ `aab`).

**Kết thúc:** `i == len(name)` đảm bảo **mọi** ký tự tên đã được gõ. ∎

## 10. Phân tích Time / Space

- **Time `O(n + m)`:** vòng `for` chạy `m` lần, mỗi lần `O(1)`; `i` chỉ tăng tối đa `n`. Thực chất `O(m)` vì `i ≤ j + 1`.
- **Space `O(1)`:** hai chỉ số.

So với cách RLE: cùng thời gian nhưng dùng `O(n + m)` bộ nhớ cho danh sách đoạn.

In [16]:
import time
name = "ab" * 500
typed = "".join(ch * 3 for ch in name)
for f in [long_pressed_rle, long_pressed_two_pointers, long_pressed_regex]:
    t0 = time.perf_counter()
    for _ in range(50):
        r = f(name, typed)
    print(f"{f.__name__:28} -> {r}, {time.perf_counter() - t0:.4f}s cho 50 lần")

long_pressed_rle             -> True, 0.0219s cho 50 lần
long_pressed_two_pointers    -> True, 0.0120s cho 50 lần
long_pressed_regex           -> True, 0.0248s cho 50 lần


## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ đề.
2. Bằng nhau hoàn toàn; 1 ký tự.
3. Ký tự **thừa ở cuối** (`alex`/`alexxr`), **thừa ở đầu** khác chữ.
4. `typed` **ngắn hơn** (`alex`/`ale`).
5. Ký tự đôi trong tên (`aab`, `saeed`, `leelee`).
6. Thứ tự đoạn khác (`ab`/`ba`).

In [17]:
tests = [
    ("alex", "aaleex", True),
    ("saeed", "ssaaedd", False),
    ("alex", "alex", True),
    ("a", "a", True),
    ("a", "aaaa", True),
    ("a", "b", False),
    ("alex", "alexxr", False),
    ("alex", "ale", False),
    ("alex", "aaleexa", False),
    ("aab", "aab", True),
    ("aab", "ab", False),
    ("leelee", "lleeelee", True),
    ("laiden", "laiden", True),
    ("ab", "ba", False),
    ("pyplrz", "ppyypllr", False),
    ("vtkgn", "vttkgnn", True),
]
solvers = [long_pressed_rle, long_pressed_groupby, long_pressed_two_pointers,
           long_pressed_run_compare, long_pressed_regex, Solution().isLongPressedName]
for name, typed, expected in tests:
    for f in solvers:
        got = f(name, typed)
        assert got == expected, (f.__name__, name, typed, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 16 test x 6 cách


In [18]:
# Ngẫu nhiên có seed: sinh typed hợp lệ bằng cách giữ phím, và typed "phá" bằng 1 thao tác
import random
rng = random.Random(925)
cnt_true = cnt_false = 0
for _ in range(600):
    name = "".join(rng.choice("ab") for _ in range(rng.randint(1, 8)))
    if rng.random() < 0.5:
        typed = "".join(c * rng.randint(1, 3) for c in name)
    else:
        typed = "".join(rng.choice("ab") for _ in range(rng.randint(1, 12)))
    expected = long_pressed_rle(name, typed)
    cnt_true += expected; cnt_false += not expected
    for f in solvers:
        assert f(name, typed) == expected, (f.__name__, name, typed)
print(f"PASS 600 test ngẫu nhiên ({cnt_true} True, {cnt_false} False)")

PASS 600 test ngẫu nhiên (286 True, 314 False)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra `name` là dãy con / quên kiểm tra phần thừa của `typed`

In [19]:
def wrong_ignore_tail(name, typed):
    i = 0
    for ch in typed:
        if i < len(name) and ch == name[i]:
            i += 1
    return i == len(name)          # không bao giờ trả False vì ký tự lạ

print("wrong_ignore_tail('alex','alexxr') =", wrong_ignore_tail("alex", "alexxr"), "| đúng: False")
print("wrong_ignore_tail('alex','aXlex') =", wrong_ignore_tail("alex", "aXlex"), "| đúng: False")

wrong_ignore_tail('alex','alexxr') = True | đúng: False
wrong_ignore_tail('alex','aXlex') = True | đúng: False


**Vì sao sai:** ký tự `typed[j]` không khớp tên thì **bắt buộc** phải là giữ phím; nếu không → `False`. **Sửa:** thêm nhánh `elif typed[j] == typed[j-1]` và `else: return False`.

### Lỗi 2: Quên kiểm tra `i == len(name)` ở cuối

In [20]:
def wrong_no_final_check(name, typed):
    i = 0
    for j in range(len(typed)):
        if i < len(name) and name[i] == typed[j]:
            i += 1
        elif j > 0 and typed[j] == typed[j - 1]:
            continue
        else:
            return False
    return True                     # quên i == len(name)

print("wrong_no_final_check('alex','aale') =", wrong_no_final_check("alex", "aale"), "| đúng: False")

wrong_no_final_check('alex','aale') = True | đúng: False


### Lỗi 3: So sánh `typed[j] == typed[j-1]` khi `j = 0`

`typed[-1]` là ký tự **cuối** chuỗi, không lỗi mà cho kết quả sai ngầm. Ví dụ `name = "b"`, `typed = "ab"`: ở `j = 0`, `typed[0] = 'a'` không khớp `'b'`, nhưng `typed[-1] = 'b'` ≠ `'a'`... trường hợp này may mắn. Với `typed = "aba"`, `name = "ba"` thì `typed[0] == typed[-1] == 'a'` → bị coi là giữ phím!

In [21]:
def wrong_no_j_guard(name, typed):
    i = 0
    for j in range(len(typed)):
        if i < len(name) and name[i] == typed[j]:
            i += 1
        elif typed[j] == typed[j - 1]:      # thiếu j > 0
            continue
        else:
            return False
    return i == len(name)

print("wrong_no_j_guard('ba','aba') =", wrong_no_j_guard("ba", "aba"), "| đúng:", long_pressed_two_pointers("ba", "aba"))

wrong_no_j_guard('ba','aba') = True | đúng: False


**Sửa:** luôn viết `j > 0 and ...` trước khi đọc `typed[j-1]`.

### Lỗi 4: So sánh tần suất bằng `Counter`

`Counter` bỏ mất **thứ tự** và **tính liên tiếp**: `name = "ab"`, `typed = "ba"` có cùng `Counter` nhưng sai. Hoặc `name = "aba"`, `typed = "aab"`.

In [22]:
from collections import Counter
def wrong_counter(name, typed):
    cn, ct = Counter(name), Counter(typed)
    return set(cn) == set(ct) and all(ct[c] >= cn[c] for c in cn)

print("wrong_counter('ab','ba') =", wrong_counter("ab", "ba"), "| đúng: False")

wrong_counter('ab','ba') = True | đúng: False


### Lỗi 5: Ưu tiên "giữ phím" trước "khớp tên"

Đã thấy ở mục 4: `wrong_priority('aab','aab')` trả `False`. **Sửa:** luôn thử khớp `name[i]` trước.

## 13. Mẹo nhận dạng pattern

- Hai chuỗi, một chuỗi là "phiên bản biến dạng" của chuỗi kia (thêm lặp, xóa, backspace...) → **hai con trỏ chạy song song**, mỗi bước phân loại ký tự.
- Đề nói "ký tự liên tiếp", "lặp lại", "giữ phím", "kéo dài" → nghĩ tới **run-length encoding** / `groupby`.
- Luôn tự hỏi: "**Khi một chuỗi hết trước** thì sao?" và "**phần dư cuối** có hợp lệ không?".

Bài liên quan: 844 Backspace String Compare, 809 Expressive Words (RLE + điều kiện ≥ 3), 392 Is Subsequence, 1446 Consecutive Characters, 443 String Compression.

## 14. Biến thể

### 14.1. LeetCode 809 — Expressive Words (kéo dài chỉ khi đoạn ≥ 3)

Một đoạn trong `s` được coi là "kéo dài" chỉ khi độ dài ≥ 3. Điều kiện cho mỗi đoạn: `ks == kw` hoặc `(ks >= 3 and ks > kw)`.

In [23]:
def is_stretchy(s: str, word: str) -> bool:
    gs, gw = rle(s), rle(word)
    if len(gs) != len(gw):
        return False
    for (cs, ks), (cw, kw) in zip(gs, gw):
        if cs != cw:
            return False
        if not (ks == kw or (ks >= 3 and ks > kw)):
            return False
    return True

s = "heeellooo"
print([w for w in ["hello", "hi", "helo"] if is_stretchy(s, w)])   # ['hello']

['hello']


### 14.2. Mỗi phím bị giữ **tối đa** `K` lần

Thêm điều kiện độ dài đoạn typed `≤ kn * K`? Chú ý: nếu tên có đoạn `k` ký tự liền nhau, mỗi ký tự có thể nhân tối đa `K` → tổng tối đa `k * K`.

In [24]:
def long_pressed_max_k(name, typed, K):
    gn, gt = rle(name), rle(typed)
    return len(gn) == len(gt) and all(cn == ct and kn <= kt <= kn * K for (cn, kn), (ct, kt) in zip(gn, gt))

print(long_pressed_max_k("alex", "aaleex", 2), long_pressed_max_k("alex", "aaaleex", 2))

True False


### 14.3. Đếm số ký tự bị "giữ phím" thêm vào

Nếu hợp lệ, số ký tự thừa chính là `len(typed) - len(name)`.

In [25]:
def extra_presses(name, typed):
    return len(typed) - len(name) if long_pressed_two_pointers(name, typed) else -1

print(extra_presses("alex", "aaleex"), extra_presses("saeed", "ssaaedd"))

2 -1


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
rle("mississippi")
[(k, len(list(g))) for k, g in groupby("aabbbaa")]
```

### Bài 2 — Tự viết hàm nén đoạn

In [26]:
def exercise_rle(s: str) -> list[tuple[str, int]]:
    # TODO: hai chỉ số i, j; j chạy tới hết đoạn giống s[i]
    pass

# assert exercise_rle("aaleex") == [('a', 2), ('l', 1), ('e', 2), ('x', 1)]
# assert exercise_rle("z") == [('z', 1)]

### Bài 3 — Tự viết lời giải hai con trỏ cho 925

In [27]:
def exercise_long_pressed(name: str, typed: str) -> bool:
    # TODO 1: i = 0
    # TODO 2: với mỗi j: khớp name? giữ phím? hay sai?
    # TODO 3: return i == len(name)
    pass

# assert exercise_long_pressed("alex", "aaleex") == True
# assert exercise_long_pressed("saeed", "ssaaedd") == False
# assert exercise_long_pressed("alex", "alexxr") == False

### Bài 4 — Is Subsequence (LeetCode 392)

`s` có phải **dãy con** (giữ thứ tự, không cần liên tiếp) của `t`?

In [28]:
def exercise_is_subsequence(s: str, t: str) -> bool:
    # TODO
    pass

# assert exercise_is_subsequence("abc", "ahbgdc") == True
# assert exercise_is_subsequence("axc", "ahbgdc") == False

### Bài 5 — String Compression dạng chuỗi

Nén `"aabcccccaaa"` thành `"a2b1c5a3"`.

In [29]:
def exercise_compress(s: str) -> str:
    # TODO
    pass

# assert exercise_compress("aabcccccaaa") == "a2b1c5a3"
# assert exercise_compress("x") == "x1"

### Bài 6 — Mini challenge: Backspace String Compare (LeetCode 844)

`#` nghĩa là xóa ký tự trước đó. Kiểm tra `s` và `t` sau khi gõ có bằng nhau không. Thử làm với **O(1) bộ nhớ** bằng hai con trỏ chạy từ cuối.

In [30]:
def exercise_backspace_compare(s: str, t: str) -> bool:
    # TODO
    pass

# assert exercise_backspace_compare("ab#c", "ad#c") == True
# assert exercise_backspace_compare("a#c", "b") == False
# assert exercise_backspace_compare("a##c", "#a#c") == True

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm từng đoạn liên tiếp | `m1 i1 s2 i1 s2 i1 p2 i1` | `groupby` tách `aa` hai lần |
| 2 | xem mục 2.2 | `while j < len(s) and s[j] == s[i]` | `groups.append((s[i], j - i)); i = j` |
| 3 | 3 nhánh | ưu tiên khớp tên | `elif j > 0 and typed[j] == typed[j-1]` |
| 4 | chỉ cần nhánh "khớp" | `i` trên s, duyệt t | `return i == len(s)` |
| 5 | dùng rle | f-string `f"{c}{k}"` | `"".join(...)` |
| 6 | đi từ cuối, đếm `skip` | gặp `#` thì `skip += 1` | hàm tìm vị trí ký tự hợp lệ kế tiếp |

## 17. Đáp án tham khảo

Bài 1: `rle("mississippi")` = `[('m',1),('i',1),('s',2),('i',1),('s',2),('i',1),('p',2),('i',1)]`; `groupby("aabbbaa")` = `[('a',2),('b',3),('a',2)]`.

In [31]:
print(rle("mississippi"))
print([(k, len(list(g))) for k, g in groupby("aabbbaa")])

[('m', 1), ('i', 1), ('s', 2), ('i', 1), ('s', 2), ('i', 1), ('p', 2), ('i', 1)]
[('a', 2), ('b', 3), ('a', 2)]


In [32]:
def answer_rle(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append((s[i], j - i))
        i = j
    return out

def answer_long_pressed(name, typed):
    i = 0
    for j in range(len(typed)):
        if i < len(name) and name[i] == typed[j]:
            i += 1
        elif j > 0 and typed[j] == typed[j - 1]:
            continue
        else:
            return False
    return i == len(name)

def answer_is_subsequence(s, t):
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)

def answer_compress(s):
    return "".join(f"{c}{k}" for c, k in answer_rle(s))

def answer_backspace_compare(s, t):
    def next_valid(x, i):
        skip = 0
        while i >= 0:
            if x[i] == "#":
                skip += 1
            elif skip > 0:
                skip -= 1
            else:
                break
            i -= 1
        return i
    i, j = len(s) - 1, len(t) - 1
    while True:
        i, j = next_valid(s, i), next_valid(t, j)
        if i < 0 or j < 0:
            return i < 0 and j < 0
        if s[i] != t[j]:
            return False
        i -= 1; j -= 1

assert answer_rle("aaleex") == [('a', 2), ('l', 1), ('e', 2), ('x', 1)]
assert answer_rle("z") == [('z', 1)]
assert answer_long_pressed("alex", "aaleex") == True
assert answer_long_pressed("saeed", "ssaaedd") == False
assert answer_long_pressed("alex", "alexxr") == False
assert answer_is_subsequence("abc", "ahbgdc") == True
assert answer_is_subsequence("axc", "ahbgdc") == False
assert answer_compress("aabcccccaaa") == "a2b1c5a3"
assert answer_compress("x") == "x1"
assert answer_backspace_compare("ab#c", "ad#c") == True
assert answer_backspace_compare("a#c", "b") == False
assert answer_backspace_compare("a##c", "#a#c") == True
for name, typed, expected in tests:
    assert answer_long_pressed(name, typed) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isLongPressedName(self, name: str, typed: str) -> bool:
        i = 0
        for j in range(len(typed)):
            if i < len(name) and name[i] == typed[j]:
                i += 1                        # ký tự tên kế tiếp
            elif j > 0 and typed[j] == typed[j - 1]:
                continue                      # giữ phím
            else:
                return False                  # ký tự lạ
        return i == len(name)                 # dùng hết tên
```

**Câu thần chú:** *"Mỗi ký tự typed: hoặc là chữ tên kế tiếp, hoặc lặp chữ vừa gõ — không thì sai; cuối cùng phải hết tên."*

**Checklist trước khi submit:**
- [ ] Ưu tiên khớp `name[i]` trước nhánh giữ phím.
- [ ] Có `i < len(name)` trước khi đọc `name[i]`.
- [ ] Có `j > 0` trước khi đọc `typed[j-1]`.
- [ ] Có `else: return False` cho ký tự lạ (kể cả thừa ở cuối).
- [ ] Cuối cùng `return i == len(name)`.
- [ ] Test: `alex/alexxr`, `alex/ale`, `aab/aab`, `saeed/ssaaedd`.